# 🤖 Gemma 4 Tech Interviewer — Colab API Server

This notebook loads your fine-tuned Gemma 4 model from Hugging Face and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `HF_TOKEN`, `NGROK_TOKEN`, and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the model.
5. Run **Cell 4** to start the API server and get your public URL.
6. Paste the URL into your backend `.env` as `GEMMA_API_URL=<url>`.

In [13]:
# CELL 1: Install dependencies
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pyngrok huggingface_hub
print('Dependencies installed!')

Dependencies installed!
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "GET /health HTTP/1.1" 200 OK
[runsync] task=open_mock_interview_session response_preview='Welcome, Waaberi. Thank you for taking the time to complete this mock interview. Our session will assess your skills in frontend development at a senior level. Please answer each question as thoroughl'
INFO:     41.78.74.12:0 - "POST /runsync HTTP/1.1" 200 OK
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
INFO:     41.78.74.12:0 - "POST /generate-question HTTP/1.1" 404 Not Found
[runsync] task=ask_technical_question response_preview='That is a set of excellent JavaScript code refactoring questions. Can you i

In [10]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found! Go to Runtime -> Change runtime type -> A100 GPU')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# CELL 3: Load fine-tuned Gemma model - LIGHTNING AI

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Get Hugging Face token from Lightning environment
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable is not set")

BASE_MODEL_ID = "google/gemma-4-e2b-it"
ADAPTER_ID = "Mohamud24/gemma-4-tech-interviewer"

# ── Precision / speed ────────────────────────────────────────────────────────
# This cell used to always load in 4-bit (bitsandbytes NF4). That halves VRAM
# but bitsandbytes 4-bit *inference* is markedly slower than plain bf16/fp16 —
# every forward pass has to dequantize weights on the fly. Measured against the
# live server while 4-bit was on: ~30s to score a short English answer and
# ~138s for a Somali one (Somali costs far more tokens per word), which is what
# made scoring feel broken.
#
# This model is small enough to run unquantized on a normal GPU, so full
# precision is now the default and 4-bit is the automatic fallback if the GPU
# cannot fit the weights. Set USE_4BIT=1 to force the old behaviour.
FORCE_4BIT = os.getenv("USE_4BIT", "0").strip().lower() in {"1", "true", "yes"}

# Use BF16 if GPU supports it, otherwise FP16
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_ID,
    token=HF_TOKEN
)


def _load_base(quantized: bool):
    kwargs = dict(
        device_map={'': 0},
        dtype=compute_dtype,
        token=HF_TOKEN,
    )
    if quantized:
        kwargs['quantization_config'] = bnb_config
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **kwargs)


if FORCE_4BIT:
    print("Loading base model in 4-bit (USE_4BIT=1)...")
    base_model = _load_base(quantized=True)
    load_mode = '4-bit (forced)'
else:
    try:
        print(f"Loading base model in {compute_dtype} (full precision, fastest)...")
        base_model = _load_base(quantized=False)
        load_mode = str(compute_dtype)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        if 'out of memory' not in str(exc).lower():
            raise
        print(f"Not enough VRAM for full precision ({exc.__class__.__name__}); falling back to 4-bit.")
        torch.cuda.empty_cache()
        base_model = _load_base(quantized=True)
        load_mode = '4-bit (fallback: out of VRAM)'

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
    token=HF_TOKEN
)

model.eval()

# Merging the LoRA weights into the base model removes the per-layer adapter
# indirection from every forward pass. It is a one-off cost here and makes each
# generated token cheaper for the whole life of the server. Skipped silently if
# the current setup does not support merging (e.g. a quantized base).
try:
    model = model.merge_and_unload()
    model.eval()
    print("LoRA adapter merged into base weights.")
except Exception as exc:
    print(f"Could not merge LoRA adapter (continuing with adapter attached): {exc}")

print(f"Model ready. Precision: {load_mode}")
if torch.cuda.is_available():
    print(f"VRAM in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
# CELL 4: Final Gemma interview serving API
# Assumes `model` and `tokenizer` were already loaded in earlier notebook cells.

import json
import os
import re
from difflib import SequenceMatcher
from threading import Lock, Thread
from typing import Any

import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ValidationError
from pyngrok import conf, ngrok
from transformers import StoppingCriteria, StoppingCriteriaList


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

NGROK_TOKEN = os.getenv("GEMMA_NGROK_TOKEN")
NGROK_DOMAIN = os.getenv("GEMMA_NGROK_DOMAIN")
MAX_INPUT_TOKENS = 4096
MODEL_LOCK = Lock()

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title="Gemma 4 Tech Interviewer API")


class InterviewRequest(BaseModel):
    endpoint: str
    payload: dict


# -----------------------------------------------------------------------------
# Small helpers
# -----------------------------------------------------------------------------

def clip(value: Any, limit: int) -> str:
    """Convert a value to compact text and hard-limit prompt size."""
    if value is None:
        return ""
    if isinstance(value, (dict, list, tuple)):
        value = json.dumps(value, ensure_ascii=False)
    return str(value).strip()[:limit]


def is_somali(language: str) -> bool:
    return str(language or "").strip().lower() in {
        "so", "somali", "so-so", "somalia"
    }


def model_to_dict(value: BaseModel) -> dict:
    if hasattr(value, "model_dump"):
        return value.model_dump()
    return value.dict()


def model_device():
    """Use the device on which the already-loaded model lives."""
    for parameter in model.parameters():
        if parameter.device.type != "meta":
            return parameter.device
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------------------------
# Efficient JSON stopping
# -----------------------------------------------------------------------------

class IncrementalJSONObjectComplete(StoppingCriteria):
    """Stop after the first complete JSON object without decoding all prior
    generated tokens again on every generation step.
    """

    def __init__(self, tokenizer, prompt_len: int):
        self.tokenizer = tokenizer
        self.last_len = prompt_len
        self.depth = 0
        self.seen_object = False
        self.in_string = False
        self.escape = False

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        current_len = input_ids.shape[1]
        if current_len <= self.last_len:
            return False

        piece = self.tokenizer.decode(
            input_ids[0][self.last_len:current_len],
            skip_special_tokens=True,
        )
        self.last_len = current_len

        for ch in piece:
            if self.escape:
                self.escape = False
                continue

            if ch == "\\" and self.in_string:
                self.escape = True
                continue

            if ch == '"':
                self.in_string = not self.in_string
                continue

            if self.in_string:
                continue

            if ch == "{":
                self.depth += 1
                self.seen_object = True
            elif ch == "}" and self.seen_object:
                self.depth -= 1
                if self.depth == 0:
                    return True

        return False


# -----------------------------------------------------------------------------
# Generation
# -----------------------------------------------------------------------------

def generate_response(
    messages: list[dict[str, str]],
    *,
    max_tokens: int,
    temperature: float = 0.0,
    stop_on_json: bool = False,
) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    device = model_device()
    encoded = {key: value.to(device) for key, value in encoded.items()}

    generation = {
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation.update(
            do_sample=True,
            temperature=temperature,
            top_p=0.90,
        )
    else:
        generation.update(do_sample=False)

    if stop_on_json:
        generation["stopping_criteria"] = StoppingCriteriaList([
            IncrementalJSONObjectComplete(
                tokenizer,
                encoded["input_ids"].shape[1],
            )
        ])

    # Prevent concurrent calls from competing for the same GPU model.
    with MODEL_LOCK, torch.inference_mode():
        output = model.generate(**encoded, **generation)

    generated = output[0][encoded["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not stop_on_json:
        # The adapter was fine-tuned on full interview transcripts, so past its
        # own single-turn reply it keeps going and hallucinates the *next* turn
        # ("\nmodel\n...", "\ncandidate: ...") instead of stopping cleanly at
        # <end_of_turn>. Confirmed live on both /ask_technical_question and
        # /open_mock_interview_session: every response was a real answer on
        # line 1 followed by a fabricated continuation, which broke the
        # backend one-question validation. The real answer is always the
        # first line, so cut there instead of chasing every role-marker string
        # the model invents.
        text = text.split("\n", 1)[0].strip()

    return text


# -----------------------------------------------------------------------------
# Structured-output parsing and validation
# -----------------------------------------------------------------------------

def parse_json_object(text: str) -> dict:
    """Extract the first JSON object. Markdown fences are tolerated."""
    cleaned = re.sub(
        r"```(?:json)?",
        "",
        text or "",
        flags=re.IGNORECASE,
    ).strip()

    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")

    value, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(value, dict):
        raise ValueError("Expected a JSON object")
    return value


def validate_json_response(
    messages: list[dict[str, str]],
    schema: type[BaseModel],
    *,
    max_tokens: int,
) -> BaseModel:
    """Generate deterministically and validate against a real schema.

    If formatting is invalid, make one deterministic repair request. The repair
    is for structure only; it must not intentionally rescore the candidate.
    """
    raw = generate_response(
        messages,
        max_tokens=max_tokens,
        temperature=0.0,
        stop_on_json=True,
    )

    try:
        return schema(**parse_json_object(raw))
    except (ValueError, json.JSONDecodeError, ValidationError):
        repair_messages = messages + [
            {"role": "assistant", "content": raw[:2500]},
            {
                "role": "user",
                "content": (
                    "The previous output did not match the required JSON schema. "
                    "Repair formatting/schema only. Preserve the same judgement. "
                    "Return one valid JSON object and nothing else."
                ),
            },
        ]

        repaired = generate_response(
            repair_messages,
            max_tokens=max_tokens,
            temperature=0.0,
            stop_on_json=True,
        )

        try:
            return schema(**parse_json_object(repaired))
        except (ValueError, json.JSONDecodeError, ValidationError) as exc:
            raise HTTPException(
                status_code=502,
                detail=f"Model returned invalid structured output: {exc}",
            )


# -----------------------------------------------------------------------------
# Output schemas
# -----------------------------------------------------------------------------

class ClarityEvaluationOutput(BaseModel):
    # ONE scoring metric only: answer clarity.
    clarityRating: int = Field(ge=0, le=10)
    feedback: str
    strengths: list[str] = Field(default_factory=list)
    improvements: list[str] = Field(default_factory=list)
    suggestedAnswer: str = ""
    needsFollowUp: bool = False
    followUpTarget: str = ""


class FeedbackOutput(BaseModel):
    detailedFeedback: str
    strengths: list[str] = Field(default_factory=list)
    improvements: list[str] = Field(default_factory=list)
    recommendations: list[str] = Field(default_factory=list)


class RoleProfileOutput(BaseModel):
    requiredSkills: list[str] = Field(default_factory=list)
    preferredSkills: list[str] = Field(default_factory=list)
    technicalStack: list[str] = Field(default_factory=list)
    responsibilities: list[str] = Field(default_factory=list)
    experienceLevel: str = ""
    candidateSkills: list[str] = Field(default_factory=list)
    candidateExperience: list[str] = Field(default_factory=list)
    candidateEducation: list[str] = Field(default_factory=list)
    candidateProjects: list[str] = Field(default_factory=list)
    candidateCertifications: list[str] = Field(default_factory=list)


# -----------------------------------------------------------------------------
# Authoritative score calculations
# -----------------------------------------------------------------------------

def clarity_rating_to_score(clarity_rating: int) -> int:
    """The LLM chooses only a small 0-10 clarity level.

    Python owns the public 0-100 scale, eliminating free-form 0-100 scoring.
    """
    rating = max(0, min(10, int(clarity_rating)))
    return rating * 10


def authoritative_overall_score(questions: list[dict]) -> int:
    """Overall score = arithmetic mean of valid stored question scores.

    No LLM is allowed to regenerate or modify this value.
    """
    scores = []

    for question in questions or []:
        try:
            score = float(question.get("score"))
        except (TypeError, ValueError):
            continue

        if 0 <= score <= 100:
            scores.append(score)

    if not scores:
        return 0

    return round(sum(scores) / len(scores))


# -----------------------------------------------------------------------------
# Interview question state / repetition control
# -----------------------------------------------------------------------------

def question_history(payload: dict, max_questions: int = 10) -> list[str]:
    questions = []

    for item in payload.get("previous_questions", []) or []:
        text = item.get("question") if isinstance(item, dict) else item
        if text:
            questions.append(clip(text, 350))

    # Backward-compatible fallback if the backend sends generic history objects.
    if not questions:
        for turn in payload.get("history", []) or []:
            if isinstance(turn, dict) and turn.get("question"):
                questions.append(clip(turn.get("question"), 350))

    return questions[-max_questions:]


def normalize_question(text: str) -> str:
    text = str(text or "").lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text


def is_near_duplicate(candidate: str, previous: list[str], threshold: float = 0.82) -> bool:
    current = normalize_question(candidate)
    if not current:
        return True

    for old in previous:
        prior = normalize_question(old)
        if not prior:
            continue
        if current == prior:
            return True
        if SequenceMatcher(None, current, prior).ratio() >= threshold:
            return True

    return False


# -----------------------------------------------------------------------------
# Prompt builders
# -----------------------------------------------------------------------------

def build_question_messages(
    payload: dict,
    *,
    rejected_question: str = "",
) -> list[dict[str, str]]:
    language = payload.get("language", "en")

    specialization = (
        payload.get("specialization")
        or payload.get("jobRole")
        or payload.get("domain")
        or "technology"
    )

    target_skill = clip(payload.get("target_skill") or payload.get("targetSkill"), 160)

    focus_skills = (
        payload.get("focus_skills")
        or payload.get("focusSkills")
        or ([target_skill] if target_skill else [])
        or []
    )

    supporting_skills = (
        payload.get("supporting_skills")
        or payload.get("supportingSkills")
        or []
    )

    data = {
        "language": "Somali" if is_somali(language) else "English",
        "specialization": specialization,
        "difficulty": payload.get("difficulty", "mid"),
        "target_skill": target_skill,
        "focus_skills": focus_skills,
        "supporting_skills": supporting_skills,
        "responsibilities": clip(payload.get("responsibilities"), 900),
        "job_context": clip(payload.get("job_description"), 1400),
        "candidate_experience": clip(payload.get("candidate_experience"), 600),
        "candidate_projects": clip(payload.get("candidate_projects"), 600),
        "previous_questions": question_history(payload),
        "previous_question": clip(payload.get("previous_question"), 350),
        "previous_answer": clip(payload.get("previous_answer"), 1000),
        "follow_up_target": clip(
            payload.get("follow_up_target") or payload.get("followUpTarget"),
            300,
        ),
        "rejected_question": clip(rejected_question, 350),
    }

    # Two grounded examples anchor the model on real Somali interview-question
    # grammar instead of letting it improvise sentence structure from scratch —
    # confirmed live that unanchored generation produces ungrammatical output
    # like "...marka aad ku dhaysto database?" (not valid Somali), even though
    # it is a complete, non-truncated sentence.
    language_rule = (
        "Write natural, grammatically correct Somali, the way a fluent native "
        "Somali speaker would actually ask a technical interview question. "
        "Keep established English technical terms such as API, React, Git, "
        "HTTP, database, server and framework names when that is the normal "
        "technical wording — do not invent Somali words for them.\n"
        "Match this grammar and tone (ask about a DIFFERENT topic, do not reuse these):\n"
        "- \"Sideed u xaqiijisaa in API-gu si sax ah u soo celiyo xogta marka user-ku codsado?\"\n"
        "- \"Waa maxay farqiga u dhexeeya SQL iyo NoSQL database-yada, tusaale ahaan?\""
        if is_somali(language)
        else
        "Write clear, natural English."
    )

    difficulty_key = str(payload.get("difficulty", "mid")).strip().lower()
    difficulty_guidance = {
        "junior": "Ask ONE simple, direct question about a single basic concept. A junior candidate must be able to answer in 1-2 short sentences. No multi-part questions, no system design, no trade-off analysis, no edge cases.",
        "mid": "Ask a practical, applied question about ONE concept a working developer handles day to day. Not a multi-part or open-ended design question.",
        "senior": "Ask a question that requires deeper reasoning: trade-offs, edge cases, or comparing approaches.",
        "lead": "Ask a broader architecture, system-design, or technical-leadership question.",
    }.get(difficulty_key, "Ask a practical, applied question about ONE concept a working developer handles day to day.")

    content = (
        "task: ask_technical_question\n"
        "Generate exactly ONE interview question from INPUT.\n\n"
        "Rules:\n"
        "- If target_skill is present, ask about EXACTLY that one skill and do not blend it with other topics.\n"
        "- focus_skills/supporting_skills are context only unless target_skill is empty.\n"
        "- Never combine multiple technologies or concepts into one long question.\n"
        "- The question must be about EXACTLY the technology named in target_skill or focus_skills — never a different, even if related, technology (example: if the topic is React, do not ask about React Native, Vue, Angular, or any other framework).\n"
        f"- Difficulty is {difficulty_key}. {difficulty_guidance}\n"
        "- If focus_skills is empty, independently choose a concrete technical topic from the selected specialization.\n"
        "- Empty optional fields must never cause a generic non-technical question.\n"
        "- If follow_up_target is present, ask one focused follow-up about that missing or unclear point.\n"
        "- Otherwise ask a new technical question on a concept not already covered by previous_questions.\n"
        "- Do not repeat or closely paraphrase any previous question.\n"
        "- Prefer direct what/how/why/comparison/trade-off questions.\n"
        "- Do not default to generic behavioral or repetitive scenario-style wording unless a scenario is genuinely necessary.\n"
        "- Vary question structure naturally across an interview.\n"
        f"- {language_rule}\n"
        "- Return only the question text. No answer, explanation, prefix, JSON, or markdown.\n\n"
        f"INPUT:\n{json.dumps(data, ensure_ascii=False)}"
    )

    return [{"role": "user", "content": content}]


def build_clarity_evaluation_messages(payload: dict) -> list[dict[str, str]]:
    language = payload.get("language", "en")

    data = {
        "language": "Somali" if is_somali(language) else "English",
        "question": clip(payload.get("question"), 1200),
        "answer": clip(payload.get("answer"), 4500),
        # Optional only. It provides context but does NOT create another score category.
        "reference_answer": clip(
            payload.get("expected_answer") or payload.get("reference_points"),
            2200,
        ),
    }

    language_rule = (
        "Write feedback in natural Somali. Keep normal English technical terms unchanged when appropriate."
        if is_somali(language)
        else
        "Write feedback in English."
    )

    content = (
        "task: score_candidate_answer\n\n"
        "Evaluate ONE thing only: the CLARITY of the candidate's spoken answer after transcription.\n"
        "Clarity means how clearly the answer responds to the question: it should be understandable, directly connected to the question, "
        "logically expressed, and sufficiently explained for the listener to understand the candidate's point.\n"
        "Judge only the clarity of the transcribed answer. Do not infer qualities that are not present in the transcript.\n"
        "Do not reward verbosity or confident wording.\n"
        "Do not penalize a candidate merely for concise wording when the meaning is clear.\n"
        "If a reference_answer is supplied, use it only to understand what the question is asking; do not require word-for-word matching.\n\n"
        "Choose clarityRating as ONE integer from 0 to 10:\n"
        "0-1 = no meaningful/understandable answer or fully unrelated response\n"
        "2-3 = very unclear; the intended answer is difficult to understand\n"
        "4-5 = partly clear but important explanation is missing or disorganized\n"
        "6-7 = generally clear and relevant with noticeable gaps\n"
        "8-9 = clear, focused and well explained with only minor improvement needed\n"
        "10 = exceptionally clear, complete and easy to follow\n\n"
        "Set needsFollowUp=true only when one short clarification could make a partially clear answer understandable or complete.\n"
        "If needsFollowUp=true, followUpTarget must name only the unclear/missing point; do not write the full follow-up question there.\n"
        "If no useful follow-up is needed, use false and an empty followUpTarget.\n\n"
        f"{language_rule}\n"
        "Keep feedback concise and grounded only in the candidate's actual answer.\n"
        "strengths: at most 2 short items.\n"
        "improvements: at most 2 short actionable items.\n"
        "suggestedAnswer: one concise clearer version/model answer.\n\n"
        "Return ONLY one valid JSON object with exactly these keys:\n"
        "clarityRating, feedback, strengths, improvements, suggestedAnswer, needsFollowUp, followUpTarget\n\n"
        f"INPUT:\n{json.dumps(data, ensure_ascii=False)}"
    )

    return [{"role": "user", "content": content}]


def build_feedback_messages(interview_data: dict, overall_score: int) -> list[dict[str, str]]:
    language = interview_data.get("language", "en")

    questions = []
    for item in interview_data.get("questions", []) or []:
        questions.append({
            "question": clip(item.get("text") or item.get("question"), 400),
            "answer": clip(item.get("userAnswer") or item.get("answer"), 900),
            "clarityScore": item.get("score"),
            "feedback": clip(item.get("feedback"), 350),
        })

    language_rule = (
        "Write all narrative feedback in natural Somali and keep normal English technical terms when appropriate."
        if is_somali(language)
        else
        "Write all narrative feedback in English."
    )

    data = {
        "overallClarityScore": overall_score,
        "questions": questions,
    }

    content = (
        "task: feedback\n\n"
        "Create the final interview feedback using INPUT.\n"
        "The interview measures ANSWER CLARITY only.\n"
        "Do not create any category scores or any new evaluation dimensions.\n"
        "The supplied overallClarityScore and per-question clarityScore values are authoritative.\n"
        "Do not recalculate, alter, estimate, or invent any score.\n"
        "Base the narrative only on the supplied questions, answers, and stored feedback.\n"
        f"{language_rule}\n"
        "LENGTH LIMITS (stay within these or the response will be rejected):\n"
        "- detailedFeedback: 150-300 characters.\n"
        "- strengths, improvements, recommendations: at most 3 items each, 40-100 characters per item.\n\n"
        "Return ONLY one valid JSON object with exactly these keys:\n"
        "detailedFeedback, strengths, improvements, recommendations\n\n"
        f"INPUT:\n{json.dumps(data, ensure_ascii=False)}"
    )

    return [{"role": "user", "content": content}]


def build_parse_messages(payload: dict) -> list[dict[str, str]]:
    data = {
        "role": clip(payload.get("role") or payload.get("jobRole") or "Technology", 200),
        "job_description": clip(payload.get("job_description"), 4000),
        "resume_text": clip(payload.get("resume_text"), 4000),
    }

    content = (
        "task: parse_role_profile\n"
        "Extract only information supported by INPUT. Never invent missing information.\n"
        "Return ONLY one valid JSON object using exactly these keys:\n"
        "requiredSkills, preferredSkills, technicalStack, responsibilities, experienceLevel, "
        "candidateSkills, candidateExperience, candidateEducation, candidateProjects, candidateCertifications\n"
        "All fields except experienceLevel are arrays of strings.\n"
        "experienceLevel must be junior, mid, senior, lead, or an empty string if unsupported.\n"
        "Use empty arrays/empty string when information is absent.\n\n"
        f"INPUT:\n{json.dumps(data, ensure_ascii=False)}"
    )

    return [{"role": "user", "content": content}]


def build_session_messages(task: str, payload: dict) -> list[dict[str, str]]:
    language = payload.get("language", "en")
    candidate = clip(payload.get("candidate_name") or "Candidate", 100)
    specialization = clip(
        payload.get("specialization") or payload.get("jobRole") or payload.get("domain") or "technology",
        300,
    )

    somali = is_somali(language)
    language_rule = (
        "Use natural, grammatically correct Somali and preserve normal English technical terms when appropriate."
        if somali else
        "Use natural English."
    )

    if task in {"open_mock_interview_session", "open_hiring_interview_session"}:
        instruction = (
            "Write one short, natural interview opening. Mention the interview area if useful. "
            "Do not ask a technical interview question yet. Do not use a fixed canned phrase."
        )
    else:
        instruction = (
            "Write one short, natural interview closing statement. Do not ask the candidate another question, "
            "do not request extra information, and do not use a repetitive fixed outro."
        )

    content = (
        f"task: {task}\n"
        f"{instruction}\n"
        f"{language_rule}\n"
        "Return only the sentence(s), with no label or markdown.\n"
        f"candidate: {candidate}\n"
        f"specialization: {specialization}"
    )

    return [{"role": "user", "content": content}]


# -----------------------------------------------------------------------------
# Handlers
# -----------------------------------------------------------------------------

def translate_question_to_somali(english_question: str) -> str:
    """Translate an already-correct English interview question into Somali.

    Composing NEW technical content and producing correct Somali grammar in
    the same generation step asks too much of this fine-tune's weaker Somali
    fluency — confirmed live with plausible-but-wrong output like "Sidee loo
    horumarisaa CSS selectors..." ("horumarisaa" = advance/develop, the wrong
    verb for "how do you use/write"). Translating a sentence whose meaning is
    already fixed and correct is a much more constrained task, and gets
    noticeably better grammatical results from smaller models than free
    composition in a lower-resource language.
    """
    messages = [{
        "role": "user",
        "content": (
            "task: translate_to_somali\n"
            "Translate the following interview question into natural, "
            "grammatically correct Somali — exactly how a fluent native "
            "Somali speaker would actually ask it out loud in a technical "
            "interview.\n"
            "Keep established English technical terms (API, React, database, "
            "server, and framework/library names) unchanged — do not invent "
            "Somali words for them.\n"
            "Return ONLY the translated question: no quotes, no explanation, "
            "no English commentary, exactly one '?' at the end.\n\n"
            f"English question: {english_question}"
        ),
    }]
    translated = generate_response(
        messages,
        max_tokens=130,
        temperature=0.30,
        stop_on_json=False,
    ).strip().strip('"')
    return translated or english_question


def handle_question(payload: dict) -> str:
    previous = question_history(payload)
    target_is_somali = is_somali(payload.get("language", "en"))

    # Always COMPOSE in English, even for a Somali interview, then translate
    # (see translate_question_to_somali above) — English composition from
    # this model is reliably correct, so pushing language handling into a
    # separate, narrower translation step avoids asking the model to
    # invent technical content and get Somali grammar right at once.
    compose_payload = dict(payload)
    if target_is_somali:
        compose_payload["language"] = "en"

    question = generate_response(
        build_question_messages(compose_payload),
        max_tokens=130,
        temperature=0.70,
        stop_on_json=False,
    ).strip()

    # One targeted regeneration when the generated question is too close to a
    # previous question. This is code-level repetition control, not only prompt
    # advice.
    if is_near_duplicate(question, previous):
        question = generate_response(
            build_question_messages(compose_payload, rejected_question=question),
            max_tokens=130,
            temperature=0.85,
            stop_on_json=False,
        ).strip()

    if not question:
        raise HTTPException(status_code=502, detail="Model returned an empty question")

    if target_is_somali:
        question = translate_question_to_somali(question)

    return question


def handle_score_candidate_answer(payload: dict) -> str:
    judged = validate_json_response(
        build_clarity_evaluation_messages(payload),
        ClarityEvaluationOutput,
        max_tokens=360,
    )

    result = model_to_dict(judged)

    # The model never outputs the public 0-100 score directly.
    # Python deterministically maps 0-10 clarityRating -> 0-100 score.
    result["score"] = clarity_rating_to_score(judged.clarityRating)

    return json.dumps(result, ensure_ascii=False)


def handle_feedback(payload: dict) -> str:
    interview_data = payload.get("interview_data", {}) or {}
    questions = interview_data.get("questions", []) or []

    overall_score = authoritative_overall_score(questions)

    # 520 was too tight: with no explicit per-field length cap in the prompt
    # the model would sometimes ramble past it and get cut off mid-string,
    # producing unparseable JSON (confirmed live: 502 "Model returned invalid
    # structured output" / "Unterminated string..." on both English and
    # Somali interviews). Paired with the new LENGTH LIMITS section above,
    # this is real headroom rather than just a bigger ceiling.
    report = validate_json_response(
        build_feedback_messages(interview_data, overall_score),
        FeedbackOutput,
        max_tokens=900,
    )

    result = model_to_dict(report)

    # Authoritative Python result; no category scores are generated.
    result["overallScore"] = overall_score

    return json.dumps(result, ensure_ascii=False)


def handle_parse(payload: dict) -> str:
    parsed = validate_json_response(
        build_parse_messages(payload),
        RoleProfileOutput,
        max_tokens=600,
    )
    return json.dumps(model_to_dict(parsed), ensure_ascii=False)


def handle_session(task: str, payload: dict) -> str:
    temperature = 0.55 if task.startswith("open_") else 0.40
    return generate_response(
        build_session_messages(task, payload),
        max_tokens=100,
        temperature=temperature,
        stop_on_json=False,
    ).strip()


# -----------------------------------------------------------------------------
# API routing
# -----------------------------------------------------------------------------

TASK_MAP = {
    "/ask_technical_question": "ask_technical_question",
    "/score_candidate_answer": "score_candidate_answer",
    "/open_mock_interview_session": "open_mock_interview_session",
    "/close_mock_interview_session": "close_mock_interview_session",
    "/open_hiring_interview_session": "open_hiring_interview_session",
    "/close_hiring_interview_session": "close_hiring_interview_session",
    "/feedback": "feedback",
    "/parse": "parse",
}


@app.get("/health")
def health():
    return {
        "status": "online",
        "model": "Mohamud24/gemma-4-tech-interviewer",
        "provider": "colab",
        "scoring": "answer_clarity_only",
    }


@app.post("/runsync")
def runsync(req: InterviewRequest):
    task_key = req.endpoint

    if task_key not in TASK_MAP:
        raise HTTPException(
            status_code=404,
            detail=f"Unknown endpoint: {task_key}",
        )

    task = TASK_MAP[task_key]

    try:
        if task == "ask_technical_question":
            response = handle_question(req.payload)
        elif task == "score_candidate_answer":
            response = handle_score_candidate_answer(req.payload)
        elif task == "feedback":
            response = handle_feedback(req.payload)
        elif task == "parse":
            response = handle_parse(req.payload)
        else:
            response = handle_session(task, req.payload)
    except HTTPException:
        raise
    except Exception as exc:
        print(f"[runsync] task={task} error={type(exc).__name__}: {exc}")
        raise HTTPException(
            status_code=500,
            detail=f"Model serving error for task '{task}'",
        ) from exc

    print(
        f"[runsync] task={task} "
        f"response_preview={response[:220]!r}"
    )

    # Keeps the response envelope compatible with the existing backend.
    return {
        "output": {
            "response": response,
            "task": task,
        }
    }


# -----------------------------------------------------------------------------
# Colab server + ngrok
# -----------------------------------------------------------------------------

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
    )


server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

if NGROK_TOKEN:
    if NGROK_DOMAIN:
        tunnel = ngrok.connect(8000, "http", domain=NGROK_DOMAIN)
    else:
        tunnel = ngrok.connect(8000, "http")

    public_url = tunnel.public_url

    print("\n==========================================")
    print("YOUR COLAB GEMMA URL IS READY!")
    print(f"URL: {public_url}")
    print("==========================================")
    print("Paste this into your backend .env file:")
    print(f"GEMMA_API_URL={public_url}")
    print("==========================================")
else:
    print("Gemma API started on port 8000.")
    print("GEMMA_NGROK_TOKEN is not set, so no ngrok tunnel was created.")
